# Chapter 12 — When to Use RL for Agents## The Decision Framework in Code[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**No GPU required. Runtime: ~15 minutes.**Interactive implementation of the Chapter 12 RL decision framework.

In [ ]:
print('Chapter 12: Practical Decision Framework')print('This notebook walks through the 6-question decision tree')print('and implements baseline comparisons.')

## 12.1 The 6-Question Decision Framework

In [ ]:
def rl_decision_framework(task_description: str) -> dict:    """    Interactive decision tree for RL vs. alternatives.    Returns recommendation and reasoning.    """    print(f'Task: {task_description}')    print('='*60)        answers = {}        q1 = input('Q1: Does the task have a measurable success signal? (yes/partial/no): ').lower()    answers['Q1_success_signal'] = q1    if q1 == 'no':        return {'recommendation': 'SFT + Prompting',                'reason': 'Cannot measure success → RL will train toward wrong thing'}        q2 = input('Q2: Does the task require 10+ sequential steps? (yes/no): ').lower()    answers['Q2_sequential'] = q2    if q2 == 'no':        return {'recommendation': 'DPO or Contextual Bandit',                'reason': 'Single-step → RL is overkill. Use DPO for preference alignment.'}        q3 = input('Q3: Have you tried SFT + prompting + DPO already? (yes/no): ').lower()    answers['Q3_tried_alternatives'] = q3    if q3 == 'no':        return {'recommendation': 'Try alternatives first',                'reason': 'Try SFT → DPO → if they plateau, then consider RL.'}        q4 = input('Q4: Do you have a safe training environment? (yes/no): ').lower()    answers['Q4_safe_env'] = q4    if q4 == 'no':        return {'recommendation': 'Offline RL (CQL/Decision Transformer)',                'reason': 'No safe env → use offline RL on logged data only.'}        q5 = input('Q5: Do you have engineering resources for RL? (yes/no): ').lower()    answers['Q5_engineering'] = q5    if q5 == 'no':        return {'recommendation': 'Use frontier model via API with strong prompting',                'reason': 'Limited resources → use pre-trained RL model (Claude/GPT-4).'}        q6 = input('Q6: Is task value high enough to justify RL cost? (yes/no): ').lower()    answers['Q6_value'] = q6    if q6 == 'no':        return {'recommendation': 'Best prompted/SFT system and accept ceiling',                'reason': 'Low value → ROI doesn\'t justify RL training cost.'}        # Check if verifiable rewards exist    q7 = input('Q7: Is the task verifiable (math/code/SQL/structured output)? (yes/no): ').lower()    if q7 == 'yes':        return {'recommendation': 'GRPO + RLVR',                'reason': 'Verifiable + all criteria met → GRPO is more efficient than PPO.'}    else:        return {'recommendation': 'PPO-RLHF with reward model',                'reason': 'Non-verifiable + all criteria met → PPO with learned reward model.'}# Demo (non-interactive)print('Running decision framework for SQL generation task...')result = {    'task': 'Generate SQL from natural language',    'answers': {        'Q1_success_signal': 'yes (execution accuracy)',        'Q2_sequential': 'no (single generation)',        'recommendation': 'DPO or verifiable reward fine-tuning (Chapter 18 approach)',        'reason': 'Single-step task → GRPO/DPO not full PPO-RLHF'    }}for k, v in result.items():    print(f'  {k}: {v}')

## 12.2 Estimating the ROI of RL Training

In [ ]:
def estimate_rl_roi(task_volume_per_day, current_success_rate, expected_improvement,                    value_per_success_usd, training_compute_cost_usd):    """    Estimate days to break even on RL training investment.    """    value_before = task_volume_per_day * current_success_rate * value_per_success_usd    value_after  = task_volume_per_day * (current_success_rate + expected_improvement) * value_per_success_usd    daily_gain   = value_after - value_before        breakeven_days = training_compute_cost_usd / daily_gain if daily_gain > 0 else float('inf')        print(f'Daily volume:         {task_volume_per_day:,} tasks')    print(f'Current success:      {current_success_rate*100:.0f}%')    print(f'Expected improvement: +{expected_improvement*100:.0f}%')    print(f'Value per success:    ${value_per_success_usd:.2f}')    print(f'Training cost:        ${training_compute_cost_usd:,.0f}')    print(f'Daily value gain:     ${daily_gain:,.2f}')    print(f'Break-even:           {breakeven_days:.0f} days')    return breakeven_days# SQL coding assistant exampleprint('=== SQL Assistant ROI Analysis ===')estimate_rl_roi(    task_volume_per_day=5000,    current_success_rate=0.73,    expected_improvement=0.08,  # 73% → 81%    value_per_success_usd=0.50,    training_compute_cost_usd=2000  # A100 GPU hours)